# Assistant IA BCM — comprendre le pipeline RAG, étape par étape

Ce notebook rejoue **tout le pipeline** de l'assistant documentaire de la
Banque Centrale de Mauritanie (BCM) : chargement des documents, indexation,
recherche, garde-fous, génération — et explique **pourquoi** chaque étape
existe, pas seulement **ce qu'elle fait**.

Il est pensé comme un notebook d'entraînement de modèle : chaque section
correspond à une phase (données → features → entraînement → inférence →
validation), avec du code exécutable et des mesures réelles à chaque étape.

**Prérequis**
- Environnement virtuel du projet (`.venv`) sélectionné comme noyau Jupyter.
- Aucune clé API n'est nécessaire pour l'essentiel du notebook : le mode de
  génération par défaut (`extractive`) ne fait aucun appel réseau. Les
  sections qui touchent à OpenAI/Gemini/Ollama sont clairement indiquées et
  protégées (elles s'exécutent seulement si un fournisseur est configuré).
- Le modèle d'embedding local (`intfloat/multilingual-e5-small`, ~470 Mo) doit
  avoir été téléchargé une première fois (voir `storage/models/`) — c'est déjà
  le cas si vous avez déjà lancé l'application une fois.

**Installer les dépendances si nécessaire :**
```bash
cd bcm_rag_chatbot
.venv/bin/pip install -r requirements-dev.txt
```

## Sommaire

1. [Vue d'ensemble & architecture](#1.-Vue-d'ensemble-%26-architecture)
2. [Mise en place](#2.-Mise-en-place)
3. [Le corpus — données brutes](#3.-Le-corpus-%E2%80%94-donn%C3%A9es-brutes)
4. [Nettoyage et découpage en passages (chunking)](#4.-Nettoyage-et-d%C3%A9coupage-en-passages-(chunking))
5. [Indexation lexicale — TF-IDF](#5.-Indexation-lexicale-%E2%80%94-TF-IDF)
6. [Le glossaire métier bilingue](#6.-Le-glossaire-m%C3%A9tier-bilingue)
7. [Indexation sémantique — embeddings locaux](#7.-Indexation-s%C3%A9mantique-%E2%80%94-embeddings-locaux)
8. [Recherche hybride & fusion RRF](#8.-Recherche-hybride-%26-fusion-RRF)
9. [Garde-fous de pertinence](#9.-Garde-fous-de-pertinence)
10. [Reformulation et reranking par LLM](#10.-Reformulation-et-reranking-par-LLM)
11. [Génération de la réponse](#11.-G%C3%A9n%C3%A9ration-de-la-r%C3%A9ponse)
12. [Contrôle post-génération](#12.-Contr%C3%B4le-post-g%C3%A9n%C3%A9ration)
13. [Analyse locale des graphiques](#13.-Analyse-locale-des-graphiques)
14. [Support bilingue français/arabe](#14.-Support-bilingue-fran%C3%A7ais%2Farabe)
15. [Multi-sources et citations](#15.-Multi-sources-et-citations)
16. [Mémoire conversationnelle](#16.-M%C3%A9moire-conversationnelle)
17. [Évaluation quantitative](#17.-%C3%89valuation-quantitative)
18. [Performance et latence](#18.-Performance-et-latence)
19. [Déploiement](#19.-D%C3%A9ploiement)
20. [Limites connues](#20.-Limites-connues)
21. [Résumé et ressources](#21.-R%C3%A9sum%C3%A9-et-ressources)

## 1. Vue d'ensemble & architecture

**En une phrase :** un pipeline RAG (Retrieval-Augmented Generation) hybride
lexical + sémantique, avec reformulation et reranking par LLM, garde-fous de
pertinence, extraction visuelle locale des graphiques, mémoire
conversationnelle et support bilingue FR/AR — exposé via une API Flask et
consommé par un widget web embarquable.

```
Site bcm.mr ──<script>── Vercel (widget statique) ──XHR──▶ Railway (API Flask + index RAG)
```

### Diagramme complet du pipeline

Ce diagramme (généré depuis `docs/diagrammes/architecture_complete.mmd`)
détaille les quatre grandes zones du système :
- **A. Préparation locale du rapport** — indexation (hors ligne, avant toute question) ;
- **B. Traitement d'une question** — tout ce que ce notebook explique en détail ;
- **C. Limites de données** — ce qui reste strictement local ;
- **D. Évaluation reproductible** — comment la qualité est mesurée et calibrée.

![Architecture complète du pipeline BCM RAG](../docs/diagrammes/architecture_complete.png)

<details>
<summary>Source Mermaid (cliquer pour déplier — utile si vous éditez le diagramme)</summary>

```mermaid
flowchart TB
    classDef user fill:#DBEAFE,stroke:#2563EB,color:#172554,stroke-width:2px
    classDef front fill:#EDE9FE,stroke:#7C3AED,color:#2E1065,stroke-width:2px
    classDef api fill:#DCFCE7,stroke:#16A34A,color:#052E16,stroke-width:2px
    classDef rag fill:#FEF3C7,stroke:#D97706,color:#451A03,stroke-width:2px
    classDef data fill:#FCE7F3,stroke:#DB2777,color:#500724,stroke-width:2px
    classDef gen fill:#E0F2FE,stroke:#0284C7,color:#082F49,stroke-width:2px
    classDef guard fill:#FEE2E2,stroke:#DC2626,color:#450A0A,stroke-width:2px

    subgraph PREP["A. Préparation locale du rapport"]
        direction LR
        PDF["Rapport BCM<br/>127 pages"]:::data
        HASH["Contrôle<br/>SHA-256"]:::data
        CLEAN["Extraction et<br/>nettoyage"]:::data
        CHUNKS["Passages textuels<br/>et lignes de tableaux"]:::data
        TFIDF["TF-IDF mots<br/>et caractères"]:::rag
        DENSE["Embeddings multilingues<br/>locaux - 384 dimensions"]:::rag
        INDEX["Index persistant<br/>2 415 passages"]:::data

        PDF --> HASH --> CLEAN --> CHUNKS
        CHUNKS --> TFIDF --> INDEX
        CHUNKS --> DENSE --> INDEX
        HASH -. "PDF modifié" .-> CLEAN
    end

    subgraph RUN["B. Traitement d'une question"]
        direction LR
        U["Utilisateur"]:::user
        GRADIO["Interface Gradio<br/>Français ou العربية"]:::front
        MEMORY["Mémoire transmise par le client<br/>isolée par conversation"]:::front
        FLASK["API Flask<br/>validation"]:::api
        LANG["Langue explicite transmise<br/>fr ou ar"]:::api
        FOLLOW{"Question liée à<br/>un échange antérieur ?"}:::guard
        FOCUS["Sujet actif, ancien tour ciblé<br/>et pages déjà citées"]:::rag
        PAGEMAP["Résolution tableau ou graphique<br/>page imprimée vers page PDF"]:::rag
        REPEAT{"Répétition<br/>exacte ?"}:::guard
        SCANNED{"Page citée<br/>peu textuelle ?"}:::guard
        DOCOCR["OCR documentaire local<br/>ordre des colonnes"]:::rag
        EXPAND["Glossaire métier<br/>sans réponse mémorisée"]:::rag
        FIRST["Recherche hybride<br/>initiale"]:::rag
        ARFAST{"Arabe et correspondance<br/>locale forte ?"}:::guard
        PLANQ{"Question courte,<br/>faible ou ambiguë ?"}:::guard
        PLANNER["Jusqu'à 4 reformulations<br/>question seule"]:::gen
        MULTI["Recherches locales<br/>par formulation"]:::rag
        FUSION["Fusion RRF<br/>12 standard - 30 graphique"]:::rag
        CHARTQ{"Question sur<br/>un graphique ?"}:::guard
        CHARTPAGE["Sélection thématique<br/>jusqu'à 30 candidats"]:::rag
        RENDER["Rendu PNG local<br/>pages ciblées à 170 dpi"]:::data
        OCR["OCR Apple Vision local<br/>libellés et coordonnées"]:::rag
        CHARTREAD["Cadrage, séries, calculs<br/>et explication citée"]:::gen
        AMBIG{"Plusieurs périmètres<br/>confirmés localement ?"}:::guard
        SUGGEST["Formulations proches<br/>tirées du rapport"]:::front
        CONFIRM["Choix de<br/>l'utilisateur"]:::user
        RELEVANT{"Preuves<br/>suffisantes ?"}:::guard
        FASTGEN{"Voie arabe<br/>rapide ?"}:::guard
        RERANK["Reranking<br/>jusqu'à 8 passages"]:::rag
        PROMPT["Contexte strict<br/>extraits uniquement"]:::gen
        ANSWER["Réponse directe,<br/>calculs et citations"]:::gen
        BIDI["Mise en forme arabe RTL<br/>nombres et citations isolés"]:::front
        DISPLAY["Réponse, pages<br/>et extraits"]:::front
        REFUS["Information absente<br/>du rapport"]:::guard
        FALLBACK["Réponse extractive<br/>de secours"]:::gen

        U --> GRADIO --> MEMORY --> FLASK --> LANG --> PAGEMAP --> FOLLOW
        FOLLOW -->|"Non"| EXPAND
        FOLLOW -->|"Oui"| FOCUS --> REPEAT
        REPEAT -->|"Oui"| DISPLAY
        REPEAT -->|"Non"| SCANNED
        SCANNED -->|"Non"| EXPAND
        SCANNED -->|"Oui"| DOCOCR --> EXPAND
        EXPAND --> FIRST --> ARFAST
        ARFAST -->|"Oui"| FUSION
        ARFAST -->|"Non"| PLANQ
        PLANQ -->|"Non"| FUSION
        PLANQ -->|"Oui"| PLANNER --> MULTI --> FUSION
        FUSION --> CHARTQ
        CHARTQ -->|"Oui"| CHARTPAGE --> RENDER --> OCR --> CHARTREAD --> BIDI
        CHARTQ -->|"Non"| AMBIG
        AMBIG -->|"Oui"| SUGGEST --> CONFIRM --> GRADIO
        AMBIG -->|"Non"| RELEVANT
        RELEVANT -->|"Non"| REFUS --> DISPLAY
        RELEVANT -->|"Oui"| FASTGEN
        FASTGEN -->|"Oui - 1 appel"| PROMPT
        FASTGEN -->|"Non"| RERANK
        RERANK -->|"Aucune preuve"| REFUS
        RERANK -->|"Passages utiles"| PROMPT --> ANSWER --> BIDI --> DISPLAY --> U
        ANSWER -. "Erreur technique" .-> FALLBACK --> BIDI
    end

    INDEX -. "Matrices, vecteurs et lignes" .-> FIRST
    INDEX -. "Même index local" .-> MULTI

    subgraph PRIVACY["C. Limites de données"]
        direction LR
        QONLY["Planification<br/>question uniquement"]:::guard
        LOCALDOC["Retrieval et embeddings<br/>rapport conservé localement"]:::guard
        EVIDENCE["Reranking et réponse<br/>candidats strictement limités"]:::guard
        LOCALCHART["Images et OCR des graphiques<br/>conservés sur la machine"]:::guard
        QONLY --> LOCALDOC --> EVIDENCE --> LOCALCHART
    end

    PLANNER -.-> QONLY
    MULTI -.-> LOCALDOC
    RERANK -.-> EVIDENCE
    OCR -.-> LOCALCHART

    subgraph EVAL["D. Évaluation reproductible"]
        direction LR
        CASES["41 cas<br/>directs, reformulés, ambigus et absents"]:::data
        METRICS["Hit@1/3/5/12, MRR<br/>acceptation et refus"]:::api
        RESULT["Hit@5 97,14 %<br/>Hit@12 100 %<br/>Refus 100 %<br/>58 tests réussis"]:::guard
        CASES --> METRICS --> RESULT
    end

    RESULT -. "calibrage" .-> EXPAND
    RESULT -. "seuil 0,88" .-> RELEVANT
```
</details>

### Composants du dépôt

| Dossier | Rôle |
|---|---|
| `api/` | API Flask : orchestration (`app.py`), index hybride (`rag.py`), fournisseurs LLM (`providers.py`), graphiques (`charts.py`), embeddings (`embeddings.py`), sources multi-documents (`sources.py`), glossaire (`query.py`) |
| `core/` | Configuration centralisée (`config.py`), langue/RTL (`language.py`), logs (`logging_config.py`) |
| `frontend/` | Interface Gradio — démo interne uniquement |
| `widget/` | Widget JS embarquable, sans dépendance |
| `data/` | Documents sources (PDF + lettres) |
| `storage/` | Index persistant, cache modèle, cache OCR graphiques |
| `scripts/` | Construction d'index, OCR, évaluation, `pipeline_walkthrough.py` (version terminal de ce notebook) |
| `evaluation/` | Jeux de questions de référence + résultats |
| `tests/` | 58 tests automatisés |

Seul `api/` + `core/` est déployé en production (voir section 19).

## 2. Mise en place

On importe directement les modules du projet — aucune réécriture : ce
notebook exécute le **vrai code** de `api/` et `core/`, pas une simulation.

La configuration (`core/config.py`) est le premier module chargé. Comme un
script d'entraînement qui valide ses hyperparamètres avant de lancer les
epochs, elle lit et valide `.env` avant toute exécution : une valeur hors
bornes arrête le programme ici plutôt que de produire un comportement
silencieusement incorrect plus tard.

In [ ]:
import sys
from pathlib import Path


def find_project_root(start: Path) -> Path:
    """Remonte l'arborescence jusqu'à trouver la racine du projet."""
    for candidate in [start, *start.parents]:
        if (candidate / "core" / "config.py").exists():
            return candidate
    raise RuntimeError("Racine du projet introuvable (core/config.py absent).")


PROJECT_ROOT = find_project_root(Path.cwd())
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "scripts"))
print(f"Racine du projet : {PROJECT_ROOT}")

In [ ]:
from core.config import get_settings

settings = get_settings()

print(f"Profil d'exécution (APP_ENV)              : {settings.app_env}")
print(f"Document source                            : {settings.report_path.name}")
print(f"Index persistant                           : {settings.index_path}")
print(f"TOP_K (passages finaux)                    : {settings.top_k}")
print(f"RETRIEVAL_CANDIDATES (candidats bruts)     : {settings.retrieval_candidates}")
print(f"SEMANTIC_RETRIEVAL activé                  : {settings.semantic_retrieval}")
print(f"SEMANTIC_WEIGHT (poids sémantique dans RRF): {settings.semantic_weight}")
print(f"MIN_RELEVANCE_SCORE (garde-fou lexical)    : {settings.min_relevance_score}")
print(f"MIN_SEMANTIC_SCORE (garde-fou sémantique)  : {settings.min_semantic_score}")
print(f"Modèle d'embedding local                   : {settings.embedding_model}")
print(f"GENERATION_PROVIDER (.env)                 : {settings.generation_provider}")

## 3. Le corpus — données brutes

Le corpus réunit **deux types de sources**, normalisées par `api/sources.py`
en une structure pivot (`Document` → plusieurs `Segment`s) :

| Source | Format d'origine | Unité citable |
|---|---|---|
| Rapport annuel 2025 | PDF natif (texte extrait par `pypdf`) | 1 page PDF = 1 segment |
| Lettres d'information (2026, janv.–juil.) | Image → PDF paginé → **OCR Apple Vision** (pré-calculé, versionné dans `data/lettres_information/ocr/`) | 1 page de lettre = 1 segment |

Le serveur de production **n'a besoin ni du moteur OCR ni des images** pour
les lettres : seuls les fichiers texte annexes sont lus. Cette séparation est
ce qui permet à l'indexation de tourner sur un serveur Linux alors que l'OCR
lui-même (Apple Vision) est macOS-only.

In [ ]:
from api.sources import load_corpus

corpus = load_corpus(settings.report_path)

total_segments = sum(len(document.segments) for document in corpus)
print(f"Documents chargés          : {len(corpus)}")
print(f"Segments (pages) au total  : {total_segments}")
print()
for document in corpus:
    print(
        f"  {document.doc_id:<28} {document.source_type:<7} "
        f"{len(document.segments):>3} page(s)  {document.title[:50]}"
    )

Chaque `Document` porte un `checksum` (empreinte SHA-256 du fichier ou du
texte OCR). C'est ce qui permet à l'index de détecter tout seul qu'un document
a changé et de se reconstruire automatiquement, sans intervention manuelle —
voir `RAGIndex.load()` à la section 5.

## 4. Nettoyage et découpage en passages (chunking)

Avant l'indexation, chaque page passe par `clean_pdf_page()` : suppression des
numéros de page isolés, du titre répété « Rapport annuel 2025 » en haut de
chaque page, et recollage des mots coupés en fin de ligne par une césure.

Regardons le texte d'une vraie page, avant/après nettoyage.

In [ ]:
from api.sources import clean_pdf_page
from pypdf import PdfReader

reader = PdfReader(str(settings.report_path))
raw_page_text = reader.pages[4].extract_text() or ""   # page PDF 5 (conjoncture macroéconomique)
cleaned_page_text = clean_pdf_page(raw_page_text)

print("── AVANT nettoyage (extrait) ──")
print(raw_page_text[:400])
print()
print("── APRÈS nettoyage (extrait) ──")
print(cleaned_page_text[:400])

Le texte nettoyé est ensuite **découpé en passages ("chunks")** par
`RAGIndex._chunk_page()` :
- taille cible ~1150 caractères, recouvrement de 180 caractères (pour ne
  jamais couper une idée en deux) ;
- découpage prioritaire aux frontières de paragraphes, puis de phrases.

Un **second découpeur dédié**, `_table_line_chunks()`, cible spécifiquement
les lignes de tableaux chiffrés : sans lui, une valeur isolée comme
« 158,7 milliards de MRU » se retrouverait diluée dans un paragraphe
générique de 1150 caractères et perdrait sa spécificité lexicale. Il détecte
les lignes à forte densité numérique, leur associe l'en-tête de colonnes/année
le plus proche et le dernier libellé rencontré.

In [ ]:
from api.rag import RAGIndex

# Utiliser une instance temporaire, non liée au fichier d'index persistant,
# uniquement pour illustrer le comportement du chunker sur une seule page.
demo_engine = RAGIndex(settings.report_path, settings.index_path)
page_chunks = demo_engine._chunk_page(cleaned_page_text, pdf_page=5, first_id=0)
table_chunks = demo_engine._table_line_chunks(cleaned_page_text, pdf_page=5, first_id=100)

print(f"Chunks 'standard' produits pour cette page : {len(page_chunks)}")
for chunk in page_chunks[:2]:
    print(f"  [{len(chunk.text):>4} caractères] {chunk.text[:140]}...")

print()
print(f"Chunks 'table_row' produits pour cette page : {len(table_chunks)}")
for chunk in table_chunks[:2]:
    print(f"  [kind={chunk.kind}] {chunk.text[:160]}...")

## 5. Indexation lexicale — TF-IDF

C'est ici que l'analogie avec l'entraînement d'un modèle devient concrète :
deux `TfidfVectorizer` de scikit-learn sont **ajustés** (`fit_transform`) sur
l'ensemble des chunks du corpus — l'équivalent du `fit()` d'un modèle
classique. Une fois ajustés, ils sont figés et réutilisés pour vectoriser
chaque nouvelle question.

Deux vectoriseurs, combinés :

| Vectoriseur | Granularité | Poids dans le score lexical | Rôle |
|---|---|---|---|
| **Mots** | uni- et bi-grammes de mots | 78 % | capte le sens porté par le vocabulaire |
| **Caractères** | n-grammes de caractères 3 à 5 | 22 % | robuste aux fautes de frappe, sigles, variantes morphologiques |

`RAGIndex.load()` charge l'index existant s'il est à jour (empreinte SHA-256
du corpus inchangée et version de schéma identique), sinon il reconstruit —
exactement comme on reprendrait un entraînement depuis un checkpoint, ou on
relancerait tout si les données ont changé.

In [ ]:
from api.rag import RAGIndex
from collections import Counter

engine = RAGIndex(settings.report_path, settings.index_path)
metadata = engine.load().metadata   # construit l'index s'il n'existe pas encore, sinon le charge

kind_counts = Counter(chunk.kind for chunk in engine.chunks)
print(f"Chunks indexés au total : {len(engine.chunks)}")
for kind, count in sorted(kind_counts.items()):
    print(f"  dont kind='{kind}' : {count}")

print()
print(f"Vocabulaire TF-IDF mots (1-2 grammes)       : {len(engine.word_vectorizer.vocabulary_):,}".replace(",", " "))
print(f"Vocabulaire TF-IDF caractères (3-5 grammes) : {len(engine.char_vectorizer.vocabulary_):,}".replace(",", " "))
print(f"Matrice mots (chunks × features)            : {engine.word_matrix.shape}")

sparsity = 100 * (1 - engine.word_matrix.nnz / (engine.word_matrix.shape[0] * engine.word_matrix.shape[1]))
print(f"Sparsité de la matrice mots                 : {sparsity:.2f} %")
print(f"Version de schéma de l'index                : {metadata.get('index_schema_version')}")

### Ce que le vectoriseur "voit" réellement dans une question

Pour rendre le TF-IDF tangible, regardons les termes qui portent le plus de
poids dans le vecteur d'une question — exactement ce que le moteur compare
ensuite à chaque chunk par produit scalaire.

In [ ]:
import numpy as np

sample_question = "Quel a été le taux de croissance du PIB réel en 2025 ?"
query_vector = engine.word_vectorizer.transform([sample_question])
feature_names = engine.word_vectorizer.get_feature_names_out()

nonzero = query_vector.tocoo()
weighted_terms = sorted(zip(nonzero.col, nonzero.data), key=lambda pair: -pair[1])

print(f"Question : « {sample_question} »")
print("Termes retenus par le vectoriseur MOTS, par poids TF-IDF décroissant :")
for col, weight in weighted_terms:
    print(f"  {feature_names[col]:<20} poids={weight:.4f}")
print()
print("Les mots vides français (« le », « du », « en »...) ont été retirés à")
print("l'entraînement (stop_words) : ils ne portent aucun poids discriminant.")

## 6. Le glossaire métier bilingue

`api/query.py` contient un petit glossaire **déterministe** (pas de LLM, pas
de réponse mémorisée) qui traduit certaines formulations utilisateur vers le
vocabulaire exact du rapport, avant même la recherche. Il gère aussi bien des
synonymes français que des formulations arabes courantes.

C'est un choix délibéré : un glossaire versionné et testable est prévisible,
contrairement à une expansion de requête laissée à un LLM à chaque appel.

In [ ]:
from api.query import build_retrieval_query

examples = [
    "coussin en devises",
    "bilan agrégé des banques",
    "crédits en difficulté",
    "السيولة المصرفية",       # liquidité bancaire
    "الناتج المحلي",           # PIB / croissance économique
]

for question in examples:
    enriched = build_retrieval_query(question)
    added = enriched[len(question):].strip()
    print(f"« {question} »")
    print(f"  → + « {added} »" if added else "  → (rien ajouté)")
    print()

## 7. Indexation sémantique — embeddings locaux

Chaque chunk est représenté par un **vecteur dense de 384 dimensions**,
produit localement par le modèle multilingue
[`intfloat/multilingual-e5-small`](https://huggingface.co/intfloat/multilingual-e5-small)
(Sentence Transformers) — **jamais envoyé à une API externe**. C'est ce qui
permet au moteur de comprendre qu'une reformulation sans vocabulaire commun
parle quand même du même sujet.

Convention du modèle E5 : les **passages** sont encodés avec le préfixe
`"passage: "`, les **questions** avec le préfixe `"query: "` — les deux rôles
ne sont pas interchangeables dans l'espace vectoriel appris par ce modèle.

### Intuition : la similarité cosinus capture le sens, pas les mots

Avant de charger l'index complet, une démonstration à trois phrases suffit à
sentir ce que le modèle "comprend" — les trois sont encodées ici avec le même
rôle (`"passage: "`) pour rester directement comparables entre elles ; la
section 8 utilisera l'encodage asymétrique question/passage réellement
employé en production (préfixe `"query: "` côté question).

In [ ]:
from api.embeddings import embed_documents

demo_sentences = [
    "Quel a été le taux de croissance du PIB réel en 2025 ?",
    "De combien l'activité économique a-t-elle progressé en volume ?",   # paraphrase
    "Quelle est la recette traditionnelle du sushi japonais ?",            # sans rapport
]
vectors = embed_documents(
    demo_sentences,
    model=settings.embedding_model,
    cache_path=settings.embedding_cache_path,
)
print(f"Forme de la matrice : {vectors.shape}  (3 phrases × 384 dimensions)")
print()

similarity_question_paraphrase = float(vectors[0] @ vectors[1])
similarity_question_unrelated = float(vectors[0] @ vectors[2])
print(f"Similarité cosinus [question PIB] vs [paraphrase PIB]      : {similarity_question_paraphrase:.4f}")
print(f"Similarité cosinus [question PIB] vs [phrase sans rapport] : {similarity_question_unrelated:.4f}")
print()
print("La paraphrase, sans un seul mot en commun avec la question d'origine,")
print("obtient une similarité plus élevée que la phrase totalement hors sujet —")
print("c'est ce signal que le retrieval hybride ajoute au TF-IDF. L'écart brut")
print("reste modeste : ce modèle compresse ses similarités près de 1.0, d'où")
print("le seuil de garde-fou élevé (0,88) discuté à la section 9.")

### Rattacher les vecteurs sémantiques à l'index

En production, chaque chunk du corpus est vectorisé une fois (à la
construction de l'index) et le résultat est persisté avec l'index lexical
dans le même fichier `.joblib`. La cellule suivante réutilise l'index déjà
présent sur disque s'il est à jour — pas de recalcul inutile.

In [ ]:
needs_semantic_build = (
    not engine.has_semantic_index
    or engine.embedding_model != settings.embedding_model
)

if needs_semantic_build and settings.semantic_retrieval:
    print(f"Vectorisation de {len(engine.chunks)} passages (peut prendre une minute la première fois)...")
    matrix = embed_documents(
        [chunk.text for chunk in engine.chunks],
        model=settings.embedding_model,
        cache_path=settings.embedding_cache_path,
        batch_size=settings.embedding_batch_size,
        show_progress=True,
    )
    engine.attach_semantic_embeddings(matrix, settings.embedding_model)
else:
    print("Index sémantique déjà présent et à jour — réutilisation, pas de recalcul.")

print(f"Dimensions du vecteur      : {engine.semantic_matrix.shape[1]}")
print(f"Vecteurs stockés           : {engine.semantic_matrix.shape[0]}")
print(f"Empreinte mémoire          : {engine.semantic_matrix.nbytes / (1024 * 1024):.1f} Mo")

## 8. Recherche hybride & fusion RRF

Le moteur dispose maintenant de deux classements possibles pour une même
question : un classement **lexical** (TF-IDF) et un classement **sémantique**
(cosinus sur les embeddings). Ces deux scores **ne sont pas sur la même
échelle** — un score TF-IDF de 0,3 et une similarité cosinus de 0,3 ne
signifient pas la même chose — donc on ne peut pas simplement les additionner.

La solution retenue est le **Reciprocal Rank Fusion (RRF)** : au lieu de
comparer des scores, on compare des **rangs**, qui sont toujours sur la même
échelle (1er, 2e, 3e...).

$$\text{rrf}(i) = \frac{1-w}{60 + \text{rang}_{\text{lexical}}(i)} + \frac{w}{60 + \text{rang}_{\text{sémantique}}(i)}$$

avec `w = SEMANTIC_WEIGHT` (0,45 par défaut). La constante 60 amortit l'écart
entre le 1er et le 2e rang : sans elle, être 1er au lieu de 2e changerait le
score de façon disproportionnée.

### Exemple jouet, à la main

Avant de lancer la vraie recherche, calculons le RRF sur un exemple minimal à
5 passages fictifs, pour rendre la formule tangible.

In [ ]:
toy_lexical_rank = {"A": 1, "B": 2, "C": 3, "D": 4, "E": 5}
toy_semantic_rank = {"A": 4, "B": 1, "C": 5, "D": 2, "E": 3}
w = settings.semantic_weight

print(f"{'Passage':<8}{'rang lexical':<14}{'rang sémantique':<18}{'score RRF':<10}")
toy_scores = {}
for passage in toy_lexical_rank:
    rrf = (1 - w) / (60 + toy_lexical_rank[passage]) + w / (60 + toy_semantic_rank[passage])
    toy_scores[passage] = rrf
    print(f"{passage:<8}{toy_lexical_rank[passage]:<14}{toy_semantic_rank[passage]:<18}{rrf:.6f}")

print()
ranked = sorted(toy_scores, key=toy_scores.get, reverse=True)
print(f"Classement final (RRF) : {' > '.join(ranked)}")
print("→ un passage moyen dans les deux classements (B, D) peut dépasser")
print("  un passage excellent dans un seul (A) : c'est l'effet recherché.")

### Sur l'index réel : trois questions représentatives

- une question **directe** (vocabulaire proche du rapport) ;
- une **paraphrase** (aucun mot en commun avec le texte source — c'est le
  sémantique qui la rattrape) ;
- une question **hors corpus** (le garde-fou, section 9, doit la refuser).

> ⚠️ **Point important, vérifié en exécutant ce notebook :** le champ `score`
> renvoyé par `engine.retrieve()` n'est **pas** le score RRF lui-même — c'est
> une moyenne pondérée lisible `(1-w)·lexical + w·sémantique`, calculée
> uniquement pour affichage. Le **classement** (l'ordre des lignes), lui, est
> bien décidé par le RRF sur les rangs. Les deux ne coïncident pas
> nécessairement : il est normal de voir le `score` affiché ne pas décroître
> strictement d'une ligne à l'autre.

In [ ]:
from api.embeddings import embed_query


def demo_retrieval(label, question, expected_pages):
    print(f"── {label} ──")
    print(f"« {question} »")
    enriched = build_retrieval_query(question)
    if enriched != question:
        print(f"  glossaire ajouté : « {enriched[len(question):].strip()[:120]} »")

    lexical_results = engine.retrieve(enriched, top_k=5, query_embedding=None)
    print("  Recherche LEXICALE seule :")
    for rank, item in enumerate(lexical_results[:3], start=1):
        print(f"    {rank}. page PDF {item['pdf_page']:<4} score={item['score']:.4f}  recouvrement={item['keyword_overlap']}")

    results_for_guard = lexical_results
    if engine.has_semantic_index and settings.semantic_retrieval:
        vector = embed_query(enriched, model=settings.embedding_model, cache_path=settings.embedding_cache_path)
        hybrid_results = engine.retrieve(enriched, top_k=5, query_embedding=vector, semantic_weight=settings.semantic_weight)
        print("  Recherche HYBRIDE (ordre décidé par RRF) :")
        for rank, item in enumerate(hybrid_results[:3], start=1):
            sem = item.get("semantic_score")
            print(f"    {rank}. page PDF {item['pdf_page']:<4} score_affiché={item['score']:.4f}  lexical={item['lexical_score']:.4f}  sémantique={sem if sem is None else f'{sem:.4f}'}")
        results_for_guard = hybrid_results

    accepted = engine.is_relevant(results_for_guard, settings.min_relevance_score, settings.min_semantic_score)
    verdict = "ACCEPTÉ" if accepted else "REFUSÉ"
    print(f"  Garde-fou de pertinence → {verdict}")
    if expected_pages:
        top_page = int(results_for_guard[0]["pdf_page"]) if results_for_guard else None
        mark = "✓" if top_page in expected_pages else "✗"
        print(f"  Vérité terrain : pages attendues {sorted(expected_pages)} — obtenu page {top_page} [{mark}]")
    print()


demo_retrieval(
    "1. Question directe",
    "Quel a été le taux de croissance du PIB réel en Mauritanie en 2025 ?",
    {5, 21, 121},
)
demo_retrieval(
    "2. Paraphrase (vocabulaire différent)",
    "De combien l'activité économique mauritanienne a-t-elle progressé en volume durant l'exercice ?",
    {5, 21, 121},
)
demo_retrieval(
    "3. Hors corpus (refus attendu)",
    "Quelle est la recette traditionnelle du sushi japonais ?",
    set(),
)

## 9. Garde-fous de pertinence

`RAGIndex.is_relevant()` décide si les preuves trouvées sont assez solides
pour **autoriser** la génération. C'est le garde-fou qui empêche l'assistant
de répondre sur un sujet absent du corpus.

Double preuve, l'une des deux suffit :
- **preuve lexicale** : score TF-IDF ≥ `MIN_RELEVANCE_SCORE` (0,075 par
  défaut) **et** au moins 2 mots-clés en commun avec la question ;
- **preuve sémantique** : similarité cosinus ≥ `MIN_SEMANTIC_SCORE` (0,88)
  avec au moins 2 mots-clés en commun, ou une similarité exceptionnellement
  forte (≥ 0,92) sans condition de mots-clés.

Ce seuil sémantique de **0,88** peut surprendre : les similarités du modèle
local restent élevées même pour une question totalement hors sujet — la
cellule ci-dessous le montre directement sur l'index réel (colonne
« sémantique », souvent au-dessus de 0,75 même sans aucun rapport avec le
corpus). Un seuil naïf plus bas — la première version testée utilisait 0,32 —
acceptait *toutes* les questions hors corpus. Le seuil actuel a été calibré
empiriquement sur le jeu d'évaluation (section 17) pour ne plus jamais
laisser passer une question absente du rapport.

In [ ]:
demo_questions = [
    ("Quel a été le taux de croissance du PIB réel en 2025 ?", True),
    ("Quelle est la recette traditionnelle du sushi japonais ?", False),
    ("Quel temps fera-t-il demain à Nouadhibou ?", False),
]

print(f"{'Question':<62}{'lexical':<9}{'sémantique':<12}{'recouv.':<9}{'verdict'}")
for question, _ in demo_questions:
    enriched = build_retrieval_query(question)
    vector = embed_query(enriched, model=settings.embedding_model, cache_path=settings.embedding_cache_path)
    results = engine.retrieve(enriched, top_k=5, query_embedding=vector, semantic_weight=settings.semantic_weight)
    accepted = engine.is_relevant(results, settings.min_relevance_score, settings.min_semantic_score)
    best = results[0]
    sem = best.get("semantic_score")
    verdict = "ACCEPTÉ" if accepted else "REFUSÉ"
    label = question if len(question) <= 60 else question[:57] + "..."
    print(f"{label:<62}{best['lexical_score']:<9.4f}{(sem or 0):<12.4f}{best['keyword_overlap']:<9}{verdict}")

## 10. Reformulation et reranking par LLM

Deux étapes optionnelles font appel à un LLM (OpenAI par défaut) — mais
seulement **quand c'est nécessaire**, pas systématiquement, pour limiter coût
et latence.

### a. Reformulation multiple (`plan_queries_openai`)

Déclenchée uniquement si la recherche initiale est fragile (score bas, faible
couverture de mots-clés, question courte peu spécifique — voir
`_needs_query_planning()` dans `api/app.py`). Le LLM reçoit **uniquement la
question**, jamais d'extrait du rapport, et produit jusqu'à 4 reformulations
autonomes. Chacune relance une recherche locale ; les listes sont fusionnées
par RRF (section 8).

```python
# api/providers.py — extrait réel
def plan_queries_openai(question: str, settings=None) -> dict:
    """Reformule uniquement la question ; aucun extrait du rapport n'est transmis."""
    response = OpenAI().responses.create(
        model=settings.openai_rerank_model,
        instructions=QUERY_PLANNER_INSTRUCTIONS,
        input=f"QUESTION UTILISATEUR :\n{question}",
        reasoning={"effort": "low"},
        max_output_tokens=1500,
        text={"format": PLANNER_SCHEMA},   # JSON strict imposé
    )
    ...
```

### b. Reranking (`rerank_openai`)

Une fois les candidats fusionnés, un second appel LLM affine la sélection
finale des passages avant rédaction — il **écarte** les correspondances
lexicales qui, à la lecture, ne répondent pas vraiment à la question. Cet
appel n'a lieu que si le garde-fou de la section 9 a déjà validé qu'il existe
des preuves suffisantes : inutile de faire relire des passages à un modèle
si le retrieval n'a rien trouvé de pertinent.

**Pourquoi un schéma JSON strict (`text.format`) sur les deux appels ?** Sans
lui, le modèle produisait un JSON invalide dans environ un cas sur deux
(accolade fermante manquante) — ce qui faisait perdre l'appel entier, son
délai et son coût, en pure perte.

### Essai en direct (seulement si une clé API est configurée)

La cellule suivante ne s'exécute réellement que si `OPENAI_API_KEY` est
présent dans `.env` — sinon elle explique pourquoi elle est sautée, sans
lever d'erreur.

In [ ]:
from api.providers import plan_queries_openai, resolve_provider

resolved_provider = resolve_provider(settings)
print(f"Provider résolu par la configuration actuelle : {resolved_provider}")

if resolved_provider == "openai":
    plan = plan_queries_openai(
        "Quelles réformes la BCM a-t-elle menées sur les systèmes de paiement ?",
        settings,
    )
    print("Reformulations proposées par le planificateur :")
    for reformulation in plan["queries"]:
        print(f"  - {reformulation}")
    print(f"Ambiguë ? {plan['ambiguous']}")
else:
    print(
        "Provider OpenAI non configuré ici : cette cellule est sautée pour ne "
        "consommer ni clé API ni quota. Configurez OPENAI_API_KEY dans .env "
        "puis relancez cette cellule pour voir un vrai plan de reformulation."
    )

## 11. Génération de la réponse

`GENERATION_PROVIDER` (`.env`) choisit le mode :

| Mode | Comportement |
|---|---|
| `auto` (défaut) | OpenAI si clé dispo → sinon Ollama local si dispo → sinon `extractive` |
| `openai` / `gemini` | LLM distant, réponse rédigée et citée |
| `ollama` | LLM local, aucune donnée envoyée à l'extérieur |
| `extractive` | pas de LLM : restitution directe des meilleurs passages (utilisé en tests, et ici par défaut) |

**Piège mesuré en production :** les tokens de raisonnement des modèles
récents (`reasoning`/`thinking`) sont **décomptés du plafond de sortie**. Un
plafond trop serré ne réduit donc pas le coût — seuls les tokens réellement
produits sont facturés — mais peut tronquer ou vider la réponse. D'où des
plafonds larges calibrés empiriquement
(`OPENAI_MAX_OUTPUT_TOKENS=3000`, `GEMINI_MAX_OUTPUT_TOKENS=8000`).

### Démonstration — mode extractif (sans clé API)

In [ ]:
from api.providers import answer_with_provider

demo_question = "Quel a été le taux de croissance du PIB réel en Mauritanie en 2025 ?"
enriched_question = build_retrieval_query(demo_question)
vector = embed_query(enriched_question, model=settings.embedding_model, cache_path=settings.embedding_cache_path)

search_results = engine.retrieve(
    enriched_question,
    top_k=settings.retrieval_candidates,
    query_embedding=vector,
    semantic_weight=settings.semantic_weight,
)
top_results = engine.decorate(search_results[: settings.top_k], "fr")
generation_context = engine.decorate(engine.expand_with_neighbors(top_results, max_results=12), "fr")

answer = answer_with_provider("extractive", demo_question, generation_context, [], settings, "fr")

print(f"Question : {demo_question}\n")
print("Réponse (mode extractif — sans LLM) :")
print(answer)
print()
print("Sources mobilisées :")
for item in top_results[:5]:
    print(f"  - {item.get('citation', f"p. PDF {item['pdf_page']}")}  (score={item['score']:.4f})")

### Avec un vrai LLM (seulement si configuré)

Même appel, mais avec le provider réellement résolu par la configuration —
sauté proprement si aucun fournisseur distant n'est disponible.

In [ ]:
if resolved_provider != "extractive":
    llm_answer = answer_with_provider(
        resolved_provider, demo_question, generation_context, [], settings, "fr"
    )
    print(f"Réponse rédigée par le provider « {resolved_provider} » :\n")
    print(llm_answer)
else:
    print(
        "Aucun LLM distant/local n'est configuré ou disponible : la section 11 "
        "ci-dessus a déjà montré le mode de repli 'extractive', qui est "
        "exactement ce que l'API en production sert aussi en cas de panne "
        "du fournisseur configuré."
    )

## 12. Contrôle post-génération

Un dernier garde-fou s'applique **après** la rédaction : une réponse qui ne
cite **aucune source** (`[p. PDF N]`) est traitée comme un refus, même si le
LLM a produit du texte. C'est le cas typique d'un modèle qui, malgré le
contexte fourni, écrit « cette information n'est pas disponible » — dans ce
cas, publier quand même les 5 passages candidats laisserait croire à tort
qu'ils soutiennent une affirmation.

```python
# api/app.py — extrait réel
cite_une_source = bool(re.search(r"\[[^\]\n]{2,80}\]", answer))
yield _evenement_reponse({
    "answer": answer,
    "sources": sources if cite_une_source else [],
    "grounded": cite_une_source,
    ...
})
```

In [ ]:
import re

cite_une_source = bool(re.search(r"\[[^\]\n]{2,80}\]", answer))
print(f"La réponse extractive de la section 11 cite-t-elle une source ? {cite_une_source}")
print("→ 'grounded' serait donc :", cite_une_source)

## 13. Analyse locale des graphiques

Chaîne entièrement **exécutée sur le serveur, sans envoi d'image à un
service externe** (`api/charts.py`) :

1. **Détection d'intention** — mots-clés (« graphique », « courbe »,
   « figure »...) ou formulations mesure+périodicité (« volume des virements
   par mois »).
2. **Recherche thématique élargie** — jusqu'à 30 candidats au lieu de 12.
3. **Sélection de page** — un numéro explicite (« graphique 23 ») est
   prioritaire ; sinon le sujet pèse plus que la simple présence du mot
   « graphique ».
4. **Rendu PDF → PNG local** (`pdftoppm`, 170 dpi).
5. **OCR local** via un petit exécutable **Swift/Apple Vision**
   (`scripts/chart_ocr.swift`) — donc macOS-dépendant pour cette étape.
6. **Cadrage** — le titre le plus proche isole la bonne figure sur une page
   qui en contient plusieurs.
7. **Lecture structurée** — années/séries/valeurs reliées par position,
   échelle reconstruite à partir de plusieurs graduations pour amortir une
   erreur OCR isolée.
8. **Explication rédigée**, citation systématique `[p. PDF N]`.

Les fonctions de **détection d'intention** sont de simples règles (regex),
donc importables et testables sans rendu PDF ni OCR :

In [ ]:
from api.charts import is_chart_question, is_chart_existence_question, explicit_chart_numbers

chart_examples = [
    "Explique le graphique 23 sur l'évolution de la liquidité bancaire.",
    "Explique le volume des virements par mois en 2025.",
    "Y a-t-il un graphique sur les achats et ventes de devises ?",
    "Quel a été le taux d'inflation en 2025 ?",   # ne déclenche pas la voie graphique
]

for question in chart_examples:
    is_chart = is_chart_question(question)
    is_existence = is_chart_existence_question(question)
    numbers = explicit_chart_numbers(question)
    print(f"« {question} »")
    print(f"    voie graphique ? {is_chart}   question d'existence ? {is_existence}   numéro(s) explicite(s) : {numbers or '—'}")

**Confidentialité :** les pages rendues et le texte OCR ne quittent jamais la
machine, et ne sont transmis à aucun fournisseur de génération distant — seul
le texte structuré déjà lu localement (années, séries, valeurs) entre dans le
prompt du LLM, jamais l'image elle-même.

**Limite connue :** cette voie interactive dépend d'Apple Vision, donc de
macOS. Les Lettres d'information contournent cette limite différemment :
leur OCR est **pré-calculé une fois** sur un poste macOS puis versionné — le
serveur Linux de production n'exécute jamais l'OCR lui-même pour elles
(section 3).

## 14. Support bilingue français/arabe

Le choix de langue est **explicite** (transmis par l'interface à chaque
appel) et prioritaire sur la détection automatique, qui ne sert qu'aux
anciens clients API n'envoyant pas ce champ.

Pour une question arabe, si le premier passage trouvé est déjà probant, le
backend passe **directement à la génération** — pas de traduction ni de
reranking séparés — ce qui réduit une question arabe courante à **un seul
appel distant** au lieu de trois.

Le rendu bidirectionnel Unicode évite qu'un navigateur inverse l'ordre d'un
nombre ou d'une citation au milieu d'une phrase arabe (RTL) :

In [ ]:
from core.language import is_arabic_text, format_arabic_bidi

sample_arabic_answer = (
    "بلغ معدل النمو 4.0% في عام 2025 مقابل 6.3% في عام 2024، وفقاً للتقرير "
    "[p. PDF 5]، وذلك بارتفاع قدره 300 000 أوقية موريتانية."
)

print(f"Contient de l'arabe ? {is_arabic_text(sample_arabic_answer)}")
print()
print("AVANT mise en forme bidi :")
print(sample_arabic_answer)
print()
print("APRÈS mise en forme bidi (isolats LRI/PDI autour des nombres/citations/unités) :")
print(format_arabic_bidi(sample_arabic_answer))
print()
print(
    "Le rendu texte brut ci-dessus paraît identique : les caractères invisibles "
    "U+2066/U+2069 ne changent rien à la copie de texte, seulement à l'ordre "
    "d'affichage calculé par le navigateur pour '300 000', '4.0%', '2025' et "
    "'[p. PDF 5]', chacun conservé comme un bloc gauche-à-droite indivisible."
)

## 15. Multi-sources et citations

Le corpus combine deux types de documents, chacun avec son propre format de
citation (`api/sources.py :: citation_label`) :

In [ ]:
from api.sources import citation_label, SOURCE_TYPE_PDF, SOURCE_TYPE_LETTRE

print("Rapport annuel, en français :", citation_label(SOURCE_TYPE_PDF, "Rapport annuel BCM", 39, "fr"))
print("Rapport annuel, en arabe    :", citation_label(SOURCE_TYPE_PDF, "Rapport annuel BCM", 39, "ar"))
print("Lettre d'information, fr    :", citation_label(SOURCE_TYPE_LETTRE, "Lettre d'information de la BCM — Mars 2026", 2, "fr"))
print("Lettre d'information, ar    :", citation_label(SOURCE_TYPE_LETTRE, "Lettre d'information de la BCM — Mars 2026", 2, "ar"))

Le repère du rapport annuel (`p. PDF N`) reste volontairement **identique**
dans les deux langues — il est toléré comme acronyme latin légitime même dans
une réponse arabe (section 14), car il figure déjà dans les réponses
servies, l'historique des conversations et les tests de non-régression.

Le TF-IDF étant **global au corpus**, ajouter les Lettres d'information a
légèrement redistribué les scores des passages du rapport existants. Un
mécanisme dédié (`_select_results` dans `api/app.py`) garantit qu'une source
n'évince jamais silencieusement l'autre à budget de passages constant.

## 16. Mémoire conversationnelle

Le backend est **stateless** : rien n'est stocké côté serveur entre deux
appels. Le client (widget ou Gradio) renvoie l'historique complet (jusqu'à
8 tours / 16 messages) à chaque requête dans le champ `history`.

```json
{
  "question": "Résume cette section.",
  "history": [
    {"role": "user", "content": "Y a-t-il un rapport d'auditeur externe ?"},
    {"role": "assistant", "content": "Oui, à la page PDF 119."}
  ]
}
```

À partir de cet historique, `api/app.py` sait :

| Fonction (privée, dans `api/app.py`) | Rôle |
|---|---|
| `_followup_markers()` | détecte une question liée à un échange antérieur (« cette section », « ce graphique », « répète »...) |
| `_followup_focus()` | retrouve le bon tour — le dernier par défaut, ou un plus ancien si un sujet est explicitement nommé |
| `_repeat_followup()` | détecte une demande de répétition **exacte** → sert la réponse mémorisée sans régénération (économie de coût et de latence) |
| `_contextualize_followup()` | transforme un suivi ambigu (« explique mieux ») en question autonome, réinjectant le sujet actif et les pages déjà citées |
| `_history_pages()` | extrait les pages `[p. PDF N]` déjà citées dans une réponse précédente, pour les ré-épingler en priorité |

Ces fonctions sont préfixées `_` (privées à `api/app.py`) car elles
travaillent main dans la main avec le reste de l'orchestrateur — elles ne
sont pas montrées ici en exécution directe pour ne pas dupliquer la
construction complète de l'application Flask dans ce notebook, mais leur
logique est directement lisible dans le fichier source.

Le bouton **Nouvelle conversation** de l'interface efface simplement ce
tableau côté client : la mémoire est propre à la conversation affichée, pas
partagée entre utilisateurs.

## 17. Évaluation quantitative

Comme la courbe de validation d'un entraînement, un jeu de **41 questions de
référence** (`evaluation/questions.jsonl`) mesure objectivement la qualité du
retrieval — 35 questions documentées (directes, paraphrases, comparaisons,
listes) et 6 questions volontairement absentes du corpus.

| Métrique | Signification |
|---|---|
| `Hit@k` | la bonne page apparaît-elle dans les k premiers résultats ? |
| `MRR` | moyenne de l'inverse du rang de la première bonne page |
| `answerable_acceptance` | le garde-fou accepte-t-il les questions réellement documentées ? |
| `grounded_hit_at_5` | pertinent **et** dans le top 5 |
| `refusal_accuracy` | les questions absentes sont-elles bien refusées ? |

In [ ]:
import evaluate_retrieval

cases = evaluate_retrieval.load_cases(PROJECT_ROOT / "evaluation" / "questions.jsonl")
print(f"{len(cases)} cas d'évaluation chargés.")

report_lexical = evaluate_retrieval.evaluate(
    engine, cases, top_k=12, min_score=settings.min_relevance_score, mode="lexical"
)
report_hybrid = evaluate_retrieval.evaluate(
    engine, cases, top_k=12, min_score=settings.min_relevance_score, mode="hybrid",
    embedding_model=settings.embedding_model, embedding_cache_path=settings.embedding_cache_path,
    semantic_weight=settings.semantic_weight, min_semantic_score=settings.min_semantic_score,
)

metrics_lexical = report_lexical["metrics"]
metrics_hybrid = report_hybrid["metrics"]

print(f"{'Métrique':<35}{'LEXICAL':<12}{'HYBRIDE':<12}")
for key, label in [
    ("hit_at_1", "Hit@1"), ("hit_at_3", "Hit@3"), ("hit_at_5", "Hit@5"),
    ("hit_at_12", "Hit@12"), ("mrr", "MRR"),
    ("answerable_acceptance", "Acceptation (documentées)"),
    ("grounded_hit_at_5", "Grounded Hit@5"),
    ("refusal_accuracy", "Refus correct (hors corpus)"),
]:
    print(f"{label:<35}{metrics_lexical[key]:<12.2%}{metrics_hybrid[key]:<12.2%}" if key != "mrr" else f"{label:<35}{metrics_lexical[key]:<12.4f}{metrics_hybrid[key]:<12.4f}")

### Visualisation — gain apporté par le signal sémantique

Un histogramme en texte (aucune dépendance graphique requise) pour comparer
visuellement les deux moteurs sur les métriques clés.

In [ ]:
def ascii_bar(value, width=40):
    filled = round(value * width)
    return "█" * filled + "░" * (width - filled)


print(f"{'Métrique':<16}{'':<2}{'barre':<42}{'valeur'}")
for key, label in [("hit_at_1", "Hit@1"), ("hit_at_3", "Hit@3"), ("hit_at_5", "Hit@5"), ("refusal_accuracy", "Refus")]:
    print(f"{label:<16}L {ascii_bar(metrics_lexical[key])} {metrics_lexical[key]:.2%}")
    print(f"{'':<16}H {ascii_bar(metrics_hybrid[key])} {metrics_hybrid[key]:.2%}")
    print()

delta_hit5 = metrics_hybrid["hit_at_5"] - metrics_lexical["hit_at_5"]
delta_refusal = metrics_hybrid["refusal_accuracy"] - metrics_lexical["refusal_accuracy"]
print(f"Gain Hit@5 apporté par le sémantique   : {delta_hit5:+.2%}")
print(f"Gain sur le refus hors corpus          : {delta_refusal:+.2%}")

### Répartition par difficulté

Le détail par catégorie de question montre où le sémantique apporte le plus :

In [ ]:
print(f"{'Difficulté':<18}{'LEXICAL':<12}{'HYBRIDE':<12}")
for difficulty in sorted(set(metrics_lexical['success_by_difficulty']) | set(metrics_hybrid['success_by_difficulty'])):
    lex = metrics_lexical['success_by_difficulty'].get(difficulty, float('nan'))
    hyb = metrics_hybrid['success_by_difficulty'].get(difficulty, float('nan'))
    print(f"{difficulty:<18}{lex:<12.2%}{hyb:<12.2%}")

Ces chiffres sont des **mesures techniques**, reproductibles à chaque
exécution de ce notebook — pas encore une validation métier BCM. Un expert
métier doit encore confirmer les pages attendues et enrichir le jeu de
questions avec des formulations réellement observées auprès des
utilisateurs.

## 18. Performance et latence

Profil mesuré en production sur une question large (voir
`README.md :: Temps de réponse`) :

| Étape | Durée |
|---|---:|
| Recherche locale (retrieval hybride) | < 0,1 s |
| Reformulation (si déclenchée) | 2,9 s |
| Reranking (si déclenché) | 2,8 s |
| Génération | 5,9 s |

**Le temps est dominé par les appels LLM distants séquentiels**, pas par la
recherche locale — ce que la section 17 confirme aussi : la recherche
hybride sur 41 questions s'exécute en une fraction de seconde au total.

Optimisations mises en place :
- **Streaming SSE** (`POST /api/ask/stream`) — le temps total ne change pas,
  mais l'utilisateur voit les premiers mots dès ~7 s au lieu d'un écran figé
  jusqu'à ~12 s. Le texte diffusé est **provisoire** : seul l'événement
  `done` a passé les contrôles de citation (section 12) et le rendu
  bidirectionnel arabe (section 14).
- **Préchargement du modèle d'embedding** au démarrage de l'application —
  économise ~6 s sur la première question de chaque worker Gunicorn.
- **Reformulation et reranking conditionnels**, jamais systématiques
  (sections 8 et 10).
- Derrière un proxy nginx, l'en-tête `X-Accel-Buffering: no` est
  indispensable : sans lui, nginx tamponne la réponse et annule tout le
  bénéfice du streaming.

## 19. Déploiement

```
Site bcm.mr ──<script src=…>──▶  Vercel   (widget statique, JS pur)
     │
     └──────────── appels XHR ───────────▶  Railway  (API Flask + index RAG)
```

- **Railway** héberge l'API Flask + l'index RAG (`api/`, `core/` uniquement —
  pas Gradio, pas le widget).
- **Vercel** héberge le widget JS statique.
- Le site bcm.mr n'a besoin que d'une balise `<script>` — le reste de la
  logique reste côté widget et API.
- **CI/CD** (GitHub Actions) : tests + lint + audit sécurité + build Docker à
  chaque push (`ci.yml`), publication d'image sur merge `main` (`cd.yml`),
  promotion manuelle vers `:production` (`promote.yml`).
- **L'index est reconstruit au build de l'image Docker** — pas de volume
  persistant : mettre à jour le corpus revient à redéployer.

Documents de référence : `DEPLOYMENT.md`, `docs/DEPLOIEMENT_RAILWAY_VERCEL.md`,
`docs/INTEGRATION_EQUIPE_BCM.md`.

## 20. Limites connues

- **OCR graphique dépendant de macOS** (Apple Vision) — la voie interactive
  d'analyse de graphiques du rapport annuel (section 13) ne peut pas tourner
  nativement sur le serveur Linux de production ; seul l'OCR pré-calculé des
  Lettres d'information est portable (section 3).
- **Mémoire conversationnelle stateless** (section 16) : tout repose sur ce
  que le client renvoie à chaque appel — une intégration cliente incomplète
  peut casser la continuité de la conversation.
- **Faux positif de clarification identifié** : un cas où le système demande
  une clarification alors qu'il avait déjà trouvé la bonne réponse ; aucun
  seuil ne distingue encore ce cas d'une ambiguïté réelle (scores mesurés
  quasi identiques : 0,455 contre 0,486).
- **Scalabilité, observabilité, gouvernance des données** : phases 4 à 6 du
  roadmap d'industrialisation, pas encore commencées.
- Le corpus reste volontairement mono-domaine (rapport + lettres) : pas de
  système multi-documents généralisé pour l'instant.

## 21. Résumé et ressources

Ce que ce notebook a montré, dans l'ordre où une question traverse
réellement le système :

1. **Configuration** validée avant tout calcul (section 2).
2. **Corpus multi-sources** normalisé (rapport PDF + lettres OCR, section 3).
3. **Chunking** en deux passes : paragraphes et lignes de tableaux (section 4).
4. **TF-IDF** mots + caractères, ajusté sur l'ensemble du corpus (section 5).
5. **Glossaire métier bilingue**, déterministe (section 6).
6. **Embeddings locaux** E5 multilingue, jamais envoyés à l'extérieur (section 7).
7. **Fusion RRF** lexical + sémantique par les rangs, pas les scores bruts (section 8).
8. **Garde-fou de pertinence** à double preuve avant toute génération (section 9).
9. **Reformulation et reranking par LLM**, déclenchés seulement si nécessaire (section 10).
10. **Génération** citée, avec repli extractif local (section 11).
11. **Contrôle post-génération** : pas de citation = pas de source publiée (section 12).
12. **Analyse locale des graphiques**, 100 % sur la machine (section 13).
13. **Bilingue FR/AR** avec rendu bidirectionnel Unicode (section 14).
14. **Citations multi-sources** cohérentes (section 15).
15. **Mémoire conversationnelle** stateless côté serveur (section 16).
16. **Qualité mesurée** de façon reproductible (section 17).

### Pour aller plus loin

- Version terminal de ce même parcours : `scripts/pipeline_walkthrough.py`
  (`.venv/bin/python scripts/pipeline_walkthrough.py --force`).
- Documentation détaillée par phase : `PHASE_1_STABILISATION.md`,
  `PHASE_2_QUALITE_RETRIEVAL.md`, `PHASE_3_ANALYSE_GRAPHIQUES.md`.
- Brief pour discussion technique : `POINT_TECH_LEAD.md`.
- Diagrammes source : `ARCHITECTURE_MERMAID.md`, `docs/diagrammes/`.
- Servir l'API réelle en local : `./run.sh`.
- Lancer les 58 tests automatisés :
  `APP_ENV=test GENERATION_PROVIDER=extractive .venv/bin/python -m pytest -q`.